# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maqsood-Ahmed110/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Ranked queue built from the honest, grouped-split model (w06) — not the
inflated random-split number. Each page gets a score, one reason code,
and an action label a human can act on directly.

Reason codes: STALE_VISIBLE, CTR_GAP, BOTH, NONE (same as w04 baseline,
now scored by the trained model rather than a fixed linear rule).
Action labels: review (score > threshold) / monitor (below threshold).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd, numpy as np, os
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

df = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position,
               MAX(report_date) AS last_report_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.content_hash_id, a.client_hash_id, c.word_count,
           a.impressions_90d, a.clicks_90d, a.avg_position,
           DATE_DIFF('day', c.content_updated_date, a.last_report_date) AS days_since_last_update
    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN agg a ON c.content_hash_id = a.content_hash_id
""").df()

df['label'] = (df['impressions_90d'] < df['impressions_90d'].median()).astype(int)
feature_cols = ['clicks_90d', 'avg_position', 'word_count', 'days_since_last_update']
model_df = df.dropna(subset=feature_cols + ['label', 'client_hash_id']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train, test = model_df.iloc[train_idx].copy(), model_df.iloc[test_idx].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(train[feature_cols], train['label'])

test['model_score'] = rf.predict_proba(test[feature_cols])[:, 1]
test['stale_flag'] = ((test['days_since_last_update'] >= 180) & (test['impressions_90d'] >= 500)).astype(int)
tier_ctr = (test['clicks_90d'] / test['impressions_90d'].replace(0, pd.NA))
test['ctr_gap_flag'] = (tier_ctr < tier_ctr.median() * 0.5).astype(int)

def reason_code(r):
    if r['stale_flag'] and r['ctr_gap_flag']: return 'BOTH'
    if r['stale_flag']: return 'STALE_VISIBLE'
    if r['ctr_gap_flag']: return 'CTR_GAP'
    return 'NONE'

test['reason_code'] = test.apply(reason_code, axis=1)
test['action'] = test['model_score'].apply(lambda s: 'review' if s > 0.5 else 'monitor')

ranked = test.sort_values('model_score', ascending=False)
print(ranked[['content_hash_id','model_score','reason_code','action']].head(10))
print("\nAction distribution:")
print(ranked['action'].value_counts())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 content_hash_id  model_score reason_code  action
192706  content_9761795d2f1672b6     0.927598        NONE  review
192508  content_6bd6b23eb02abe1b     0.925514        NONE  review
192056  content_8e3cfb0ff6a0a426     0.913872     CTR_GAP  review
176262  content_78df8a597985a72a     0.903997        NONE  review
26362   content_e242b00e2f8db731     0.882384     CTR_GAP  review
26228   content_5177ec549232369b     0.867573     CTR_GAP  review
156756  content_8766f4dd83d21f91     0.841788     CTR_GAP  review
10143   content_5d9c5b86895959de     0.835496        NONE  review
84884   content_0c074d35c64af4e4     0.828148     CTR_GAP  review
84943   content_69a32e086d1037c4     0.828074     CTR_GAP  review

Action distribution:
action
monitor    22451
review       190
Name: count, dtype: int64


## 2. Intended use and limits

Intended use: a weekly-capacity content team uses this ranked queue to
pick which ~50 pages to review first — a prioritization aid, not an
autonomous action system.

Limits: valid only for the client population and month (2026-03) this
model was trained on. Precision is honest under a grouped-by-client
split (~0.59, per w06), not the inflated random-split number (~0.74).
Does not generalize confidently to a brand-new client with no history
in this warehouse — w06's audit showed real client-specific pattern
leakage even after the fix.

## 3. Human review + the no-go list
Before acting on ANY "review" label, a human must check: is this page
intentionally evergreen/rarely-changing (a no-go for refresh flags)?
Is the low CTR due to a seasonal dip rather than genuine decline?

NO-GO — must NOT be automated:
- Auto-publishing content changes based on the score alone
- Auto-deprioritizing/removing pages flagged "monitor" — absence of a
  flag is not evidence a page is fine, only that it didn't clear this
  specific rule's threshold
- Treating the model's score as a client-facing deliverable without a
  human reviewing the specific reason code first

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

Retrain triggers: precision on a fresh grouped holdout drops meaningfully
below the ~0.59 baseline established in w06; the client mix changes
substantially (new clients added who look nothing like March 2026's
training population); or more than ~3 months pass since the training
window, since search behavior and client content strategy drift.

Monitoring: track weekly what fraction of "review" actions a human
actually confirms as correct (a simple thumbs up/down log) — a sustained
drop in that agreement rate is an early warning before the metric itself
visibly degrades.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

Exporting the ranked queue CSV (not committed, per repo's leak-guard)
and a summary metrics JSON (committed — this is the paper's receipt).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

ranked.to_csv('work/outputs/action_playbook_queue.csv', index=False)

import json
metrics = {
    "model": "RandomForestClassifier",
    "split": "GroupShuffleSplit by client_hash_id",
    "honest_precision_grouped_split": 0.587629,
    "honest_f1_grouped_split": 0.203209,
    "naive_precision_random_split": 0.744589,
    "training_month": "2026-03",
    "action_distribution": ranked['action'].value_counts().to_dict(),
    "note": "grouped-split numbers are the decision-support figures; random-split numbers are shown only to illustrate the generalization gap, per w06 audit."
}
with open('work/outputs/w07_playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported queue CSV and metrics JSON.")
print(json.dumps(metrics, indent=2))


Exported queue CSV and metrics JSON.
{
  "model": "RandomForestClassifier",
  "split": "GroupShuffleSplit by client_hash_id",
  "honest_precision_grouped_split": 0.587629,
  "honest_f1_grouped_split": 0.203209,
  "naive_precision_random_split": 0.744589,
  "training_month": "2026-03",
  "action_distribution": {
    "monitor": 22451,
    "review": 190
  },
  "note": "grouped-split numbers are the decision-support figures; random-split numbers are shown only to illustrate the generalization gap, per w06 audit."
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.